# Fake News Detection System

In [1]:
import pandas as pd

In [2]:
import re
import nltk

In [3]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

In [6]:
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,confusion_matrix)

In [7]:
df = pd.read_csv("dataset/fake_news.csv")

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [8]:
print(df.head())

   news_id                                    headline  \
0        1              Global economy faces recession   
1        2   Earthquake hits Chittagong, panic spreads   
2        3  Breakthrough in Bangladesh's tech industry   
3        4     Dengue situation worsens in the country   
4        5              New virus outbreak sparks fear   

                                           body_text      source label  
0  The Prime Minister attended the inauguration c...         CNN  Real  
1  Government sources state that an official anno...  Daily Star  Real  
2  Local administration has visited the spot for ...         BBC  Fake  
3  The Prime Minister attended the inauguration c...         BBC  Fake  
4  Government sources state that an official anno...     Reuters  Real  


In [9]:
print("Dataset shape:", df.shape)

Dataset shape: (2212, 5)


In [10]:
print("Column names:")
print(df.columns)

Column names:
Index(['news_id', 'headline', 'body_text', 'source', 'label'], dtype='str')


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2212 entries, 0 to 2211
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   news_id    2212 non-null   int64
 1   headline   2212 non-null   str  
 2   body_text  2212 non-null   str  
 3   source     2212 non-null   str  
 4   label      2212 non-null   str  
dtypes: int64(1), str(4)
memory usage: 86.5 KB


In [12]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
news_id      0
headline     0
body_text    0
source       0
label        0
dtype: int64


In [13]:
print("Duplicate rows:",df.duplicated().sum())

Duplicate rows: 0


In [14]:
df["headline"] = df["headline"].fillna("")

df["body_text"] = df["body_text"].fillna("")


print("Missing text values handled.")

Missing text values handled.


In [15]:
df = df.drop_duplicates()

print("Duplicate records removed.")

print("Dataset shape:", df.shape)

Duplicate records removed.
Dataset shape: (2212, 5)


In [16]:
df["content"] = (df["headline"] + " " + df["body_text"])

print(df[["content", "label"]].head())

                                             content label
0  Global economy faces recession The Prime Minis...  Real
1  Earthquake hits Chittagong, panic spreads Gove...  Real
2  Breakthrough in Bangladesh's tech industry Loc...  Fake
3  Dengue situation worsens in the country The Pr...  Fake
4  New virus outbreak sparks fear Government sour...  Real


In [17]:
print(df["label"].value_counts())

label
Fake    1110
Real    1102
Name: count, dtype: int64


In [18]:
df["label"] = df["label"].map({"Fake": 0,"Real": 1})

print("Labels converted.")

Labels converted.


In [19]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
stop_words = set(stopwords.words("english"))

stemmer = PorterStemmer()

print("Stop words and stemmer ready.")

Stop words and stemmer ready.


In [21]:
def clean_text(text):
    text = text.lower()

    text = re.sub(r"[^a-zA-Z]"," ",text)

    words = text.split()

    words = [
        stemmer.stem(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [22]:
df["content"] = df["content"].apply(clean_text)

print(df[["content", "label"]].head())

                                             content  label
0  global economi face recess prime minist attend...      1
1  earthquak hit chittagong panic spread govern s...      1
2  breakthrough bangladesh tech industri local ad...      0
3  dengu situat worsen countri prime minist atten...      0
4  new viru outbreak spark fear govern sourc stat...      1


In [23]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df["content"])

y = df["label"]

print("TF-IDF feature extraction completed.")

TF-IDF feature extraction completed.


In [24]:
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (2212, 100)
Target shape: (2212,)


In [25]:
X_train, X_test, y_train, y_test = train_test_split( X,y,test_size=0.20,random_state=42,stratify=y)


print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (1769, 100)
Testing data: (443, 100)


In [26]:
logistic_model = LogisticRegression(max_iter=1000)

logistic_model.fit(X_train,y_train)

print("Logistic Regression trained.")

Logistic Regression trained.


In [27]:
naive_bayes_model = MultinomialNB()

naive_bayes_model.fit(X_train,y_train)

print("Naive Bayes trained.")

Naive Bayes trained.


In [28]:
svm_model = SVC()

svm_model.fit(X_train,y_train)

print("SVM trained.")

SVM trained.


In [29]:
logistic_pred = logistic_model.predict(X_test)

naive_bayes_pred = naive_bayes_model.predict(X_test)

svm_pred = svm_model.predict(X_test)

print("Predictions completed.")

Predictions completed.


In [30]:
print("ACCURACY ")

print("Logistic Regression:",accuracy_score(y_test, logistic_pred))

print("Naive Bayes:",accuracy_score(y_test, naive_bayes_pred))

print("SVM:",accuracy_score(y_test, svm_pred))

ACCURACY 
Logistic Regression: 0.4853273137697517
Naive Bayes: 0.48306997742663654
SVM: 0.5033860045146726


In [31]:
print(" PRECISION ")

print("Logistic Regression:",precision_score(y_test,logistic_pred,zero_division=0))

print("Naive Bayes:",precision_score( y_test,naive_bayes_pred,zero_division=0))

print("SVM:",precision_score(y_test,svm_pred,zero_division=0))

 PRECISION 
Logistic Regression: 0.4847161572052402
Naive Bayes: 0.4827586206896552
SVM: 0.5021459227467812


In [32]:
print("RECALL ")

print("Logistic Regression:",recall_score(y_test,logistic_pred,zero_division=0))


print("Naive Bayes:",recall_score(y_test,naive_bayes_pred,zero_division=0))

print("SVM:",recall_score(y_test,svm_pred,zero_division=0))

RECALL 
Logistic Regression: 0.502262443438914
Naive Bayes: 0.5067873303167421
SVM: 0.5294117647058824


In [33]:
print(" F1 SCORE ")

print("Logistic Regression:",f1_score(y_test,logistic_pred,zero_division=0))

print("Naive Bayes:",f1_score(y_test,naive_bayes_pred,zero_division=0))

print("SVM:",f1_score(y_test,svm_pred,zero_division=0))

 F1 SCORE 
Logistic Regression: 0.49333333333333335
Naive Bayes: 0.49448123620309054
SVM: 0.5154185022026432


In [34]:
print(" CONFUSION MATRICES ")

print("Logistic Regression:")

print(confusion_matrix(y_test,logistic_pred))


print("\nNaive Bayes:")

print(confusion_matrix(y_test,naive_bayes_pred))

print("\nSVM:")

print(confusion_matrix(y_test,svm_pred))

 CONFUSION MATRICES 
Logistic Regression:
[[104 118]
 [110 111]]

Naive Bayes:
[[102 120]
 [109 112]]

SVM:
[[106 116]
 [104 117]]


In [35]:
logistic_accuracy = accuracy_score(y_test,logistic_pred)


naive_bayes_accuracy = accuracy_score(y_test,naive_bayes_pred)


svm_accuracy = accuracy_score(y_test, svm_pred)



print("Logistic Regression:",logistic_accuracy)

print("Naive Bayes:",naive_bayes_accuracy)

print("SVM:",svm_accuracy)

Logistic Regression: 0.4853273137697517
Naive Bayes: 0.48306997742663654
SVM: 0.5033860045146726


In [37]:
accuracies = {
    "Logistic Regression": logistic_accuracy,
    "Naive Bayes": naive_bayes_accuracy,
    "SVM": svm_accuracy
}



best_model_name = max(accuracies,key=accuracies.get)


print("Best Model:", best_model_name)
print("Best Accuracy:",accuracies[best_model_name])

Best Model: SVM
Best Accuracy: 0.5033860045146726


In [38]:
import joblib
import os

In [39]:
if best_model_name == "Logistic Regression":
    best_model = logistic_model


elif best_model_name == "Naive Bayes":
    best_model = naive_bayes_model


else:
    best_model = svm_model

In [40]:
os.makedirs("model",exist_ok=True)


joblib.dump(best_model,"model/fake_news_model.pkl")


joblib.dump(vectorizer,"model/tfidf_vectorizer.pkl")


print("Best model saved successfully.")
print("TF-IDF vectorizer saved successfully.")

Best model saved successfully.
TF-IDF vectorizer saved successfully.
